# IQM hardware usage 

This tutorial shows how to run the QAOA solver on an IQM Resonance quantum processor (the`garnet` backend will be used as a example) with QHyper: we build a small MaxCutProblem instance, run QAOA, optimize the angles on a simulator, connect to the IQM hardware and evaluate the measured results.

## Defining the problem

We create a small graph Erdős-Rényi random graph `G(n, 1/2)` and wrap its edge list in a `MaxCutProblem`

In [89]:
import networkx as nx
from QHyper.problems.maxcut import MaxCutProblem

n = 3
g = nx.erdos_renyi_graph(n, 0.5, seed=100) 
edges = list(g.edges())

problem = MaxCutProblem(edges=edges)
print(f"{n} nodes, {len(edges)} edges")
print("edges:", edges)

3 nodes, 2 edges
edges: [(0, 1), (0, 2)]


## Tuning the angles on a simulator first

Running the optimizer directly on hardware is expensive. We can optimize the QAOA angles on a local simulator (`use_simulator=True`) and then run the optimized angles once on the `IQM machine`.

In [87]:
from QHyper.solvers.gate_based.iqm import QAOA
from QHyper.optimizers import OptimizationParameter, Dummy
from QHyper.optimizers.scipy_minimizer import ScipyOptimizer

sim_solver = QAOA(
    problem,
    layers=1,
    gamma=OptimizationParameter(min=[0.0], init=[0.0], max=[6.28]),
    beta=OptimizationParameter(min=[0.0], init=[0.0], max=[6.28]),
    optimizer=ScipyOptimizer(verbose=True, method='Powell'),
    use_simulator=True,
    shots=1000,
)
optimized_results = sim_solver.solve()
optimized_results.params

Step 1/200: -1.043
Step 2/200: -1.165
Step 3/200: -1.213
Step 4/200: -1.214
Step 5/200: -1.198
Success: True. Message: Optimization terminated successfully.


{'gamma': array([5.20127485]), 'beta': array([4.97249217])}

In [92]:
from QHyper.util import sort_solver_results

def print_results(results):
    sorted_results = sort_solver_results(results.probabilities)

    print(sorted_results.dtype.names)
    for result in sorted_results:
        print(result)

In [ ]:
print_results(optimized_results)

('x0', 'x1', 'x2', 'probability')
(1, 0, 0, 0.277)
(0, 1, 1, 0.267)
(0, 0, 1, 0.124)
(1, 0, 1, 0.121)
(0, 1, 0, 0.108)
(1, 1, 0, 0.087)
(1, 1, 1, 0.009)
(0, 0, 0, 0.007)


## Running QAOA on the IQM hardware

The IQM QAOA solver connects to the hardware as soon as it is constructed: the `backend_token` is exported as the `IQM_TOKEN` environment variable (or directly in QAOA instance).

`solver.solve()` submits the QAOA circuit to the `IQM machine` for `shots` measurements and returns a `SolverResult`: `probabilities`. `params` holds the angles used.

<div class="alert alert-warning">
To run the problem on the QPU you need an IQM API token. Obtain one at <a href="https://resonance.meetiqm.com/">IQM Resonance</a> and replace the <code>&lt;YOUR_IQM_RESONANCE_TOKEN&gt;</code> placeholder. </div>

In [ ]:
hardware_solver = QAOA(
    problem,
    layers=1,
    gamma=OptimizationParameter(init=[5.20127485]),
    beta=OptimizationParameter(init=[4.97249217]),
    optimizer=Dummy(),
    backend_url="https://cocos.resonance.meetiqm.com/garnet",
    backend_token="<YOUR_IQM_RESONANCE_TOKEN>",
    shots=1000,
)
hardware_results = hardware_solver.solve()

print_results(hardware_results)

('x0', 'x1', 'x2', 'probability')
(0, 1, 1, 0.304)
(1, 0, 0, 0.239)
(0, 0, 1, 0.111)
(0, 1, 0, 0.11)
(1, 0, 1, 0.09)
(1, 1, 0, 0.089)
(0, 0, 0, 0.046)
(1, 1, 1, 0.011)


## Configuration via YAML

The same experiment can be described in a configuration file and built with `solver_from_config`. Gate-based solvers use a `device:` block: the solver `name` must be `qaoa` and `platform` must be `iqm`

### Simulator (YAML)

Local Qiskit Aer simulator (`device.type: simulator`).

In [ ]:
import yaml
from QHyper.solvers import solver_from_config

simulator_config_yaml = """
problem:
    type: MaxCutProblem
    edges: [[0, 1], [0, 2]]
solver:
    name: qaoa
    category: gate_based
    platform: iqm
    layers: 1
    gamma:
        init: [5.20127485]
    beta:
        init: [4.97249217]
    optimizer:
        type: dummy
    shots: 1000
    device:
        type: simulator
        name: iqm.resonance
"""

config = yaml.safe_load(simulator_config_yaml)
solver = solver_from_config(config)
yaml_sim_results = solver.solve()

print_results(yaml_sim_results)

('x0', 'x1', 'x2', 'probability')
(0, 1, 1, 0.297)
(1, 0, 0, 0.29)
(1, 0, 1, 0.106)
(0, 0, 1, 0.099)
(0, 1, 0, 0.094)
(1, 1, 0, 0.093)
(0, 0, 0, 0.012)
(1, 1, 1, 0.009)


### IQM hardware (YAML)

The `garnet` QPU (`device.type: qpu`, `backend: garnet`), the `token` goes inside the `device` block.

In [ ]:
garnet_config_yaml = """
problem:
    type: MaxCutProblem
    edges: [[0, 1], [0, 2]]
solver:
    name: qaoa
    category: gate_based
    platform: iqm
    layers: 1
    gamma:
        init: [5.20127485]
    beta:
        init: [4.97249217]
    optimizer:
        type: dummy
    shots: 1000
    device:
        type: qpu
        name: iqm.resonance
        backend: garnet
        token: "<YOUR_IQM_RESONANCE_TOKEN>"
"""

config = yaml.safe_load(garnet_config_yaml)
solver = solver_from_config(config)
yaml_hw_results = solver.solve()

print_results(yaml_hw_results)

('x0', 'x1', 'x2', 'probability')
(0, 1, 1, 0.305)
(1, 0, 0, 0.257)
(0, 0, 1, 0.109)
(1, 0, 1, 0.105)
(1, 1, 0, 0.089)
(0, 1, 0, 0.084)
(0, 0, 0, 0.035)
(1, 1, 1, 0.016)
